In [1]:
# Import Libraries

from ultralytics import YOLO
import supervision as sv
import cv2
import math

In [2]:
model = YOLO('yolov8s.pt')
v_path = 'SampleVideo_LowQuality.mp4'
cap = cv2.VideoCapture(v_path)

In [3]:
tracker = sv.ByteTrack()
box_annotator = sv.BoxAnnotator()
prev_centers = {}
speed_dict = {}
fps = cap.get(cv2.CAP_PROP_FPS)
pixel_to_meter = 0.05

In [4]:
speed_limit = 90


while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)[0]

    detections = sv.Detections.from_ultralytics(results)

    tracked_detections = tracker.update_with_detections(detections)

    frame = box_annotator.annotate(frame, tracked_detections)

    overspeed_count = 0  

    for box in range(len(tracked_detections)):
        x1, y1, x2, y2 = tracked_detections.xyxy[box].astype(int)
        class_id = tracked_detections.class_id[box]
        track_id = tracked_detections.tracker_id[box]

        cx = int((x1 + x2)/2)
        cy = int((y1 + y2)/2)

        if track_id in prev_centers:
            px, py = prev_centers[track_id]
            pixel_distance = math.sqrt((cx - px)**2 + (cy - py)**2)
            meter_distance = pixel_distance * pixel_to_meter
            speed_mps = meter_distance * fps
            speed_kmh = speed_mps * 3.6
            speed_dict[track_id] = speed_kmh
        else:
            speed_dict[track_id] = 0

        prev_centers[track_id] = (cx, cy)
        speed = speed_dict[track_id]

        label = f"{model.names[class_id]} {track_id} {speed:.1f} km/h"

        if speed > speed_limit:
            color = (0, 0, 255) 
            overspeed_count += 1
        else:
            color = (0, 255, 0)  
        
        cv2.putText(frame, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    cv2.putText(frame, f"Overspeed Cars: {overspeed_count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    
    cv2.imshow("YOLO Speed Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 2 persons, 21 cars, 1 motorcycle, 371.0ms
Speed: 9.8ms preprocess, 371.0ms inference, 14.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 22 cars, 1 motorcycle, 292.3ms
Speed: 6.6ms preprocess, 292.3ms inference, 6.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 21 cars, 1 motorcycle, 1 truck, 282.6ms
Speed: 9.1ms preprocess, 282.6ms inference, 3.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 20 cars, 1 motorcycle, 1 truck, 285.1ms
Speed: 9.2ms preprocess, 285.1ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 18 cars, 1 motorcycle, 1 truck, 284.7ms
Speed: 7.3ms preprocess, 284.7ms inference, 3.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 18 cars, 1 motorcycle, 277.7ms
Speed: 9.7ms preprocess, 277.7ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 18 cars, 1 motorcycle, 284.9ms
Speed: